# SymboLM — Cross-Domain 200-Question Benchmark Evaluation

**Author & Principal Architect:** Sanish Gyawali  
**AI Systems Exoskeleton:** Antigravity (Google DeepMind)  
**Evaluated Policy:** Stage 4 Symbolic Efficiency & Hard EOS Adapter (SE-HEA)  
**Target Hardware:** Kaggle Tesla T4 GPU (CUDA)  
**Repository:** [https://github.com/gyawalisanish0/symboLM](https://github.com/gyawalisanish0/symboLM)  

---  
### Evaluation Domains (200 Questions):
1. **Physics & Engineering:** Mechanics, Kinematics, RLC Circuits, Thermodynamics, Optics (30)
2. **Computer Science & Discrete Systems:** Algorithms, Big-O, Graphs, Boolean SAT, Automata (30)
3. **Chemistry & Quantitative Biology:** Stoichiometry, Buffer pH, Gas Laws, Genetics (25)
4. **Probability, Statistics & Finance:** Bayes, Expected Value, Annuities, Compounding (25)
5. **Pure & Competition Mathematics:** Number Theory, Polynomials, Geometric Invariants (40)
6. **Scientific & Multi-Hop Factual Retrieval:** Zero-Shot Bypass routing validation (25)
7. **Strategic Intent & Pragmatics:** Executive communications, crisis de-escalation (25)

In [ ]:
# Cell 1: Environment & Hardware Verification
import os
import sys
import torch

print('Python version:', sys.version)
print('Torch version :', torch.__version__)
if torch.cuda.is_available():
    print(f'[OK] NVIDIA CUDA detected! Found {torch.cuda.device_count()} GPU(s):')
    for i in range(torch.cuda.device_count()):
        mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({mem:.1f} GB VRAM)')
else:
    print('[Warning] No CUDA GPU found!')


In [ ]:
# Cell 2: Dependencies & Clean Environment
!pip install -q --upgrade pip
!pip uninstall -y -q torchao
!pip install -q transformers peft accelerate sympy datasets huggingface_hub


In [ ]:
# Cell 3: Clone latest SymboLM repository with cross-domain benchmark
!rm -rf /kaggle/working/symboLM
!git clone https://github.com/gyawalisanish0/symboLM.git /kaggle/working/symboLM
print('[OK] Successfully cloned latest SymboLM repository!')


In [ ]:
# Cell 4: Mount Stage 4 Policy Adapter
import os
import shutil
from pathlib import Path

target_dir = Path('/kaggle/working/symboLM/checkpoints/stage4_efficient_adapter')
target_dir.mkdir(parents=True, exist_ok=True)

found = False
# 1. Search for stage4_adapter in mounted kernel sources
for p in Path('/kaggle/input').glob('**/stage4_adapter'):
    if (p / 'adapter_config.json').exists():
        print(f'[OK] Found Stage 4 adapter at: {p}')
        for item in p.iterdir():
            if item.is_file():
                shutil.copy2(item, target_dir / item.name)
        found = True
        break

# 2. Search for stage 4 tarball archive
if not found:
    for arc in Path('/kaggle/input').glob('**/*stage4*policy*.tar.gz'):
        print(f'[OK] Found Stage 4 tar archive at: {arc}. Unpacking...')
        os.system(f'tar -xzf "{arc}" -C "{target_dir}"')
        found = True
        break

print('[Status] Validating Stage 4 Adapter Files:')
for f in sorted(target_dir.iterdir()):
    sz = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f'  {f.name} ({sz:.2f} MB)' if f.is_file() else f'  {f.name}/')

assert (target_dir / 'adapter_config.json').exists(), 'CRITICAL: adapter_config.json missing!'
assert (target_dir / 'adapter_model.safetensors').exists(), 'CRITICAL: adapter_model.safetensors missing!'
print('[VERIFIED] Stage 4 Efficient Policy Adapter is ready for cross-domain evaluation!')


In [ ]:
# Cell 5: Run Cross-Domain 200 Evaluation with Hard EOS Token Budget
!CUDA_VISIBLE_DEVICES=0 python /kaggle/working/symboLM/eval/run_200_mixed_benchmark.py \
  --base_model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
  --adapter_path /kaggle/working/symboLM/checkpoints/stage4_efficient_adapter \
  --data_path /kaggle/working/symboLM/data/eval_200_cross_domain_benchmark.jsonl \
  --output_path /kaggle/working/eval_cross_domain_results.json \
  --report_path /kaggle/working/EVAL_CROSS_DOMAIN_REPORT.md \
  --max_tokens 128 \
  --temperature 0.2


In [ ]:
# Cell 6: Display Cross-Domain Benchmark Scorecard
rep_path = '/kaggle/working/EVAL_CROSS_DOMAIN_REPORT.md'
if os.path.exists(rep_path):
    with open(rep_path, 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print('[Warning] Report file not found!')
